# SigLIP 入门教程

SigLIP (Sigmoid Loss for Language-Image Pre-training) 是 Google 提出的改进版 CLIP 模型。

## 学习目标

- 理解 SigLIP 与 CLIP 的区别
- 掌握 Sigmoid 损失函数的原理
- 学会使用 SigLIP 进行图像-文本匹配

## 目录

1. [什么是 SigLIP](#1-什么是-siglip)
2. [模型架构](#2-模型架构)
3. [Sigmoid vs Softmax 损失](#3-sigmoid-vs-softmax-损失)
4. [代码实践](#4-代码实践)
5. [总结](#5-总结)

## 1. 什么是 SigLIP

### 背景

CLIP 使用 **Softmax 损失** (InfoNCE) 进行对比学习，但存在一些问题：
- 需要全局归一化，计算开销大
- 难以扩展到超大 batch size

### SigLIP 的改进

SigLIP 使用 **Sigmoid 损失**，将多分类问题转化为多个二分类问题：
- 每个图像-文本对独立判断是否匹配
- 不需要全局归一化
- 支持更大的 batch size，训练更稳定

In [ ]:
# 环境准备
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

torch.manual_seed(42)

## 2. 模型架构

SigLIP 的架构与 CLIP 类似，包含：
- **视觉编码器**: 基于 ViT，使用全局平均池化（无 CLS token）
- **文本编码器**: Transformer，使用 EOS token 作为句子表示
- **SwiGLU 激活**: 比 GELU 效果更好

In [ ]:
from siglip import SigLIPConfig, SigLIP, create_siglip_model

# 创建小型模型用于演示
config = SigLIPConfig(
    image_size=224,
    patch_size=16,
    vision_layers=4,
    vision_width=256,
    vision_heads=4,
    text_layers=4,
    text_width=256,
    text_heads=4,
    embed_dim=256,
    vocab_size=32000,
    use_swiglu=True  # SigLIP 特有
)

model = SigLIP(config).to(device)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 测试前向传播
batch_size = 4
images = torch.randn(batch_size, 3, 224, 224).to(device)
input_ids = torch.randint(0, 32000, (batch_size, 32)).to(device)

with torch.no_grad():
    img_feat, txt_feat, scale, bias = model(images, input_ids)

print(f'图像特征: {img_feat.shape}')
print(f'文本特征: {txt_feat.shape}')
print(f'温度参数: {scale.item():.2f}')
print(f'偏置参数: {bias.item():.2f}')

## 3. Sigmoid vs Softmax 损失

### Softmax 损失 (CLIP)

```
L = -log(exp(s_ii) / Σ_j exp(s_ij))
```

需要对所有样本计算 softmax，是**全局**操作。

### Sigmoid 损失 (SigLIP)

```
L = -Σ_ij [y_ij * log(σ(z_ij)) + (1-y_ij) * log(1-σ(z_ij))]
```

每个样本对独立计算，是**局部**操作。

In [ ]:
from siglip import siglip_loss

# 模拟特征
img_feat = F.normalize(torch.randn(8, 256), dim=-1)
txt_feat = F.normalize(torch.randn(8, 256), dim=-1)
scale = torch.tensor(10.0)
bias = torch.tensor(-10.0)

# 计算 SigLIP 损失
loss = siglip_loss(img_feat, txt_feat, scale, bias)
print(f'SigLIP 损失: {loss.item():.4f}')

In [ ]:
# 可视化两种损失的区别
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Softmax 损失示意
axes[0].set_title('Softmax 损失 (CLIP)', fontsize=12)
axes[0].text(0.5, 0.7, '需要全局归一化', ha='center', fontsize=11)
axes[0].text(0.5, 0.5, 'exp(s_ii) / Σ exp(s_ij)', ha='center', fontsize=10, family='monospace')
axes[0].text(0.5, 0.3, '计算复杂度: O(N²)', ha='center', fontsize=10)
axes[0].axis('off')

# Sigmoid 损失示意
axes[1].set_title('Sigmoid 损失 (SigLIP)', fontsize=12)
axes[1].text(0.5, 0.7, '独立二分类', ha='center', fontsize=11)
axes[1].text(0.5, 0.5, 'σ(z_ij) 独立计算', ha='center', fontsize=10, family='monospace')
axes[1].text(0.5, 0.3, '支持更大 batch size', ha='center', fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. 代码实践

### 4.1 计算图像-文本相似度

In [ ]:
# 计算相似度矩阵
with torch.no_grad():
    similarity = model.get_similarity(img_feat, txt_feat)

print('相似度矩阵:')
print(similarity.numpy())

### 4.2 使用预定义模型

In [ ]:
# 创建不同大小的模型
for size in ['small', 'base']:
    m = create_siglip_model(size)
    params = sum(p.numel() for p in m.parameters())
    print(f'{size}: {params/1e6:.1f}M 参数')

## 5. 总结

### SigLIP 的优势

| 特性 | CLIP | SigLIP |
|------|------|--------|
| 损失函数 | Softmax | Sigmoid |
| 归一化 | 全局 | 局部 |
| 大 batch | 困难 | 容易 |
| 训练稳定性 | 一般 | 更好 |

### 适用场景

- 大规模预训练
- 分布式训练
- 图像-文本检索
- 零样本分类